# ISIC19 EDF Learnable Fusion

Kaggle notebook wrapper for the existing file-wise training code. Attach your project repository as a Kaggle input dataset, upload/copy it into `/kaggle/working/Classification`, or set `GITHUB_REPO_URL` in the setup cell. The notebook downloads the ISIC19 image dataset with KaggleHub and runs `src/main.py` directly so the training logic stays in your project files.

In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
import sys

KAGGLE_INPUT = Path('/kaggle/input')
KAGGLE_WORKING = Path('/kaggle/working')
PROJECT_DIR = KAGGLE_WORKING / 'Classification'

# If this notebook is already running inside the project directory, use it.
if (Path.cwd() / 'src' / 'main.py').exists():
    PROJECT_DIR = Path.cwd()

# Optional manual settings. Usually leave these blank.
# CODE_INPUT_DIR example: '/kaggle/input/classification-code/Classification'
# GITHUB_REPO_URL example: 'https://github.com/<user>/<repo>.git'
CODE_INPUT_DIR = os.environ.get('CODE_INPUT_DIR', '').strip()
GITHUB_REPO_URL = os.environ.get('GITHUB_REPO_URL', '').strip()

def copy_project_from_input(src_dir, dst_dir):
    src_dir = Path(src_dir)
    if not (src_dir / 'src' / 'main.py').exists():
        raise FileNotFoundError(f'No src/main.py found under {src_dir}')
    if dst_dir.exists():
        return dst_dir
    shutil.copytree(src_dir, dst_dir)
    return dst_dir

if CODE_INPUT_DIR and not (PROJECT_DIR / 'src' / 'main.py').exists():
    PROJECT_DIR = copy_project_from_input(CODE_INPUT_DIR, PROJECT_DIR)

def find_project_code_in_kaggle_input():
    if not KAGGLE_INPUT.exists():
        return None

    patterns = [
        '*/src/main.py',
        '*/*/src/main.py',
        '*/*/*/src/main.py',
    ]
    for pattern in patterns:
        for main_file in KAGGLE_INPUT.glob(pattern):
            return main_file.parent.parent

    for main_file in KAGGLE_INPUT.rglob('src/main.py'):
        return main_file.parent.parent

    return None

if not (PROJECT_DIR / 'src' / 'main.py').exists():
    found_project = find_project_code_in_kaggle_input()
    if found_project is not None:
        PROJECT_DIR = copy_project_from_input(found_project, PROJECT_DIR)

if not (PROJECT_DIR / 'src' / 'main.py').exists() and GITHUB_REPO_URL:
    subprocess.check_call(['git', 'clone', '--depth', '1', GITHUB_REPO_URL, str(PROJECT_DIR)])

if not (PROJECT_DIR / 'src' / 'main.py').exists():
    raise FileNotFoundError(
        'Could not find project code. Attach your repository as a Kaggle input dataset, '
        'upload/copy it to /kaggle/working/Classification, set CODE_INPUT_DIR to the '
        'folder containing src/main.py, or set GITHUB_REPO_URL to clone it.'
    )

sys.path.insert(0, str(PROJECT_DIR))
print('Project:', PROJECT_DIR)

In [ ]:
import importlib.util
import subprocess

if importlib.util.find_spec('kagglehub') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'kagglehub[pandas-datasets]'])

import kagglehub

ISIC19_DATASET = 'sujansarkar/isic-2019-multi-modal-classification'
downloaded_root = Path(kagglehub.dataset_download(ISIC19_DATASET))
print('Downloaded dataset root:', downloaded_root)

def is_isic19_dir(path):
    path = Path(path)
    return (
        (path / 'Train_Validation_dataset').exists()
        and (path / 'Test_dataset').exists()
    )

def find_isic19_dir(root):
    root = Path(root)
    candidates = [root, root / 'isic19']
    candidates.extend(path for path in root.rglob('*') if path.is_dir() and path.name.lower() == 'isic19')
    candidates.extend(path.parent for path in root.rglob('Train_Validation_dataset') if path.is_dir())

    seen = set()
    for path in candidates:
        resolved = path.resolve()
        if resolved in seen:
            continue
        seen.add(resolved)
        if is_isic19_dir(path):
            return path

    raise FileNotFoundError(
        f'Could not find ISIC19 folders inside {root}. Expected Train_Validation_dataset/ and Test_dataset/.'
    )

ISIC19_SOURCE_DIR = find_isic19_dir(downloaded_root)

# src/data/dataset_config.py expects CLASSIFICATION_DATA_ROOT/isic19.
# Make a lightweight symlink in /kaggle/working instead of copying the images.
DATA_ROOT = KAGGLE_WORKING / 'data'
DATA_ROOT.mkdir(parents=True, exist_ok=True)
ISIC19_LINK = DATA_ROOT / 'isic19'
if ISIC19_LINK.is_symlink() or ISIC19_LINK.is_file():
    ISIC19_LINK.unlink()
elif ISIC19_LINK.exists():
    shutil.rmtree(ISIC19_LINK)
ISIC19_LINK.symlink_to(ISIC19_SOURCE_DIR, target_is_directory=True)

os.environ['CLASSIFICATION_DATA_ROOT'] = str(DATA_ROOT)

print('ISIC19 source:', ISIC19_SOURCE_DIR)
print('ISIC19 link:', ISIC19_LINK)
print('CLASSIFICATION_DATA_ROOT:', os.environ['CLASSIFICATION_DATA_ROOT'])

In [ ]:
# Main experiment controls. Defaults are for ISIC19 + EDF learnable fusion.
DATASET = 'isic19'
MODEL = 'edf'
BACKBONE = 'resnet101'
EXPERT_MODE = 'multi_layer'
DISAGREEMENT_TYPE = 'learnable'

EPOCHS = 100
FREEZE_EPOCHS = 5
BATCH_SIZE = 16
LR = 1e-4
LOSS = 'focal'
SCHEDULER = 'cosine'
IMG_SIZE = 224
PROJ_DIM = 224
NUM_WORKERS = 2
SEED = 42
USE_AMP = True

OUTPUT_DIR = '/kaggle/working/outputs'

In [ ]:
import subprocess

cmd = [
    sys.executable, str(PROJECT_DIR / 'src' / 'main.py'),
    '--output-dir', OUTPUT_DIR,
    '--dataset', DATASET,
    '--model', MODEL,
    '--scheduler', SCHEDULER,
    '--backbone1', BACKBONE,
    '--expert-mode', EXPERT_MODE,
    '--disagreement-type', DISAGREEMENT_TYPE,
    '--proj-dim', str(PROJ_DIM),
    '--epochs', str(EPOCHS),
    '--loss', LOSS,
    '--batch-size', str(BATCH_SIZE),
    '--lr', str(LR),
    '--img-size', str(IMG_SIZE),
    '--num-workers', str(NUM_WORKERS),
    '--seed', str(SEED),
    '--freeze-epochs', str(FREEZE_EPOCHS),
]

if USE_AMP:
    cmd.append('--amp')

print('Running:')
print(' '.join(cmd))

env = os.environ.copy()
subprocess.run(cmd, cwd=PROJECT_DIR, env=env, check=True)

In [ ]:
from pathlib import Path

results_dir = Path(OUTPUT_DIR) / 'results' / DATASET
print('Results directory:', results_dir)
for path in sorted(results_dir.glob('*')):
    print(path)